# Fase 2 - VQR com Data Re-uploading

**Arquitetura:** 6 qubits | 6 camadas | 3 re-uploading layers | AmplitudeEmbedding (64 features)

**Referência:** Pérez-Salinas et al. (2020) — *Data re-uploading for a universal quantum classifier*. Quantum, 4, 226.

**Motivação:** Os dados do DF mostram Rt variando de 0.53 a 2.76, com transições abruptas entre regimes. Re-uploading força o circuito a reprocessar as features em múltiplas profundidades, capturando essas mudanças de regime.

In [1]:
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri("mlruns")
    _MLFLOW = False  # tracking desativado (entregável)
except ImportError:
    _MLFLOW = False
    print("[AVISO] mlflow nao instalado — execute: pip install mlflow")
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from feature_engineering import construir_features, splits_validacao

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)

dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}

CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {
        "X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
        "X_test":  np.array(te["X"]), "y_test":  np.array(te["y"]),
        "datas":   te["datas"], "nome": desc,
        "periodo_treino": split["periodo_treino"],
        "periodo_teste":  split["periodo_teste"],
    }

print(f"Features ({len(dataset['feature_names'])}): {dataset['feature_names']}")
for nome, d in CENARIOS.items():
    print(f"{nome}: treino={len(d['X_train'])} | teste={len(d['X_test'])} | "
          f"target_max={max(d['y_test']):.0f}")

# ── utilitários compartilhados (utils_qml.py na raiz do projeto) ─────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from utils_qml import (calcular_wis, metricas, salvar_padrao, plot_pred,
                        validar_json_saida, validar_pipeline,
                        testar_invariancia_quantica, testar_propriedades,
                        validar_golden)
testar_propriedades()
print("[OK] utils_qml importado")

c:\Users\julia\anaconda3\envs\qml_dengue\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features (13): ['casos_est_lag1', 'casos_est_lag2', 'casos_est_lag3', 'casos_est_lag4', 'Rt_lag1', 'Rt_lag2', 'p_rt1_lag1', 'receptivo_lag1', 'transmissao_lag1', 'tempmed_lag1', 'umidmed_lag1', 'SE_sin', 'SE_cos']
C1: treino=36 | teste=143 | target_max=25714
C2: treino=88 | teste=91 | target_max=25714
C3: treino=125 | teste=54 | target_max=947
[HYPOTHESIS] Biblioteca nao instalada. Executando versao simplificada.
             Para instalar: pip install hypothesis
[PROP OK] 500 combinacoes aleatorias: WIS>=0, RMSE>=0, MAE>=0 em todos.
[OK] utils_qml importado


In [2]:
# ── Validação do pipeline de dados (integração) ──────────────────────────────
validar_pipeline(dataset, splits)


[PIPELINE OK] 13 features | 4 splits | serie=179 semanas | cenarios C1/C2/C3 prontos


True

In [3]:
import pennylane as qml
from pennylane import numpy as pnp
from copy import deepcopy
from sklearn.preprocessing import MinMaxScaler

CONFIG = {
    "n_qubits": 6, "n_layers": 6, "n_features": 64,
    "reuploading_layers": 3,
    "n_epochs": 80, "lr_init": 0.05, "lr_min": 0.001,
    "batch_size": 16, "n_bootstrap": 5,
    "seed": 42, "patience": 15,
}
np.random.seed(CONFIG["seed"])
print(f"PennyLane: {qml.__version__}")
print(f"Config: {CONFIG['n_qubits']}q | {CONFIG['n_layers']}L | {CONFIG['reuploading_layers']} re-up | B={CONFIG['n_bootstrap']}")

# AmplitudeEmbedding exige 2^n_qubits = 64 features → normaliza para [0.01, 0.99]
CENARIOS_F2 = {}
for cen, d in CENARIOS.items():
    n_feat = CONFIG["n_features"]
    sc = MinMaxScaler(feature_range=(0.01, 0.99))
    X_tr_all = d["X_train"]
    # padding se n_features > colunas disponíveis
    if X_tr_all.shape[1] < n_feat:
        pad = n_feat - X_tr_all.shape[1]
        X_tr_all = np.hstack([X_tr_all, np.zeros((X_tr_all.shape[0], pad))])
        X_te_all = np.hstack([d["X_test"], np.zeros((d["X_test"].shape[0], pad))])
    else:
        X_tr_all = X_tr_all[:, :n_feat]
        X_te_all = d["X_test"][:, :n_feat]
    X_tr = sc.fit_transform(X_tr_all)
    X_te = sc.transform(X_te_all)
    CENARIOS_F2[cen] = {**d, "X_train": X_tr, "X_test": X_te, "scaler": sc}
print("[OK] Dados preparados para AmplitudeEmbedding (16 features, [0.01,0.99])")

PennyLane: 0.40.0
Config: 6q | 6L | 3 re-up | B=5
[OK] Dados preparados para AmplitudeEmbedding (16 features, [0.01,0.99])


In [4]:
try:
    _dev_name = "lightning.qubit"
    import pennylane_lightning  # noqa
except ImportError:
    _dev_name = "default.qubit"
    print("[AVISO] pennylane-lightning nao instalado, usando default.qubit (mais lento)")
n_q = CONFIG["n_qubits"]
dev = qml.device(_dev_name, wires=n_q)

@qml.qnode(dev)
def vqr_f2(x, var_w, reup_w):
    """
    VQR Fase 2: AmplitudeEmbedding + StronglyEntanglingLayers + Data Re-uploading.
    Re-uploading: re-injeta x após camadas variacionais para capturar não-linearidades.
    """
    n_reup = CONFIG["reuploading_layers"]
    n_pure = CONFIG["n_layers"] - n_reup

    qml.AmplitudeEmbedding(x, wires=range(n_q), normalize=True)
    if n_pure > 0:
        qml.StronglyEntanglingLayers(var_w[:n_pure], wires=range(n_q))
    for i in range(n_reup):
        # Re-uploading: rotações parametrizadas pelos dados
        for q in range(n_q):
            qml.RY(x[q % len(x)] * reup_w[i, q, 0] + reup_w[i, q, 1], wires=q)
            qml.RZ(x[(q+1) % len(x)] * reup_w[i, q, 2], wires=q)
        qml.StronglyEntanglingLayers(var_w[n_pure+i:n_pure+i+1], wires=range(n_q))
    return [qml.expval(qml.PauliZ(q)) for q in range(n_q)]

n_var  = CONFIG["n_layers"] * n_q * 3
n_reup = CONFIG["reuploading_layers"] * n_q * 3
print(f"Parâmetros: variacionais={n_var} | re-uploading={n_reup} | output={n_q+1}")
print(f"Total: {n_var + n_reup + n_q + 1}")

Parâmetros: variacionais=108 | re-uploading=54 | output=7
Total: 169


In [5]:
if "x_viz" not in dir() or "w_viz" not in dir():
    import numpy as _np
    _rng = _np.random.RandomState(0)
    from pennylane import numpy as _pnp
    _nq  = CONFIG.get("n_qubits", 6) if "CONFIG" in dir() else 6
    _nl  = CONFIG.get("n_layers", 4) if "CONFIG" in dir() else 4
    _nr  = CONFIG.get("reuploading_layers", 1) if "CONFIG" in dir() else 1
    _amp = _rng.uniform(0.1, 1.0, 2 ** _nq)
    x_viz = _pnp.array(_amp / _np.linalg.norm(_amp))
    w_viz = _pnp.array(_rng.uniform(-_np.pi, _np.pi, (_nl, _nq, 3)))
    reup_viz = _pnp.array(_rng.uniform(-0.5, 0.5, (_nr, _nq, 3)))
if "reup_viz" not in dir():
    import numpy as _np
    from pennylane import numpy as _pnp
    _nr = CONFIG.get("reuploading_layers", 1) if "CONFIG" in dir() else 1
    _nq = CONFIG.get("n_qubits", 6) if "CONFIG" in dir() else 6
    reup_viz = _pnp.array(_np.random.RandomState(0).uniform(-0.5, 0.5, (_nr, _nq, 3)))

def _vqr_f2_invariancia(x, var_w):
    return vqr_f2(x, var_w, reup_viz)

# ── Invariância quântica (determinismo, bounds, shape) ───────────────────────
testar_invariancia_quantica(
    circuit_fn=_vqr_f2_invariancia,
    n_qubits=CONFIG.get('n_qubits', 6),
    x_sample=x_viz,
    weights_sample=w_viz,
    contexto="Fase2_VQR_ReUploading"
)

[QUANTICO OK] [Fase2_VQR_ReUploading] Determinismo / bounds [-1,1] / shape(6) — OK


True

In [6]:
class VQRFase2:
    def __init__(self, cfg=CONFIG):
        self.cfg = cfg
        self.n_q = cfg["n_qubits"]
        self.n_l = cfg["n_layers"]
        self.n_r = cfg["reuploading_layers"]

    def _init(self, rng):
        var_w  = pnp.array(rng.uniform(-np.pi, np.pi, (self.n_l, self.n_q, 3)), requires_grad=True)
        reup_w = pnp.array(rng.uniform(-0.5, 0.5,    (self.n_r, self.n_q, 3)), requires_grad=True)
        out_sc = pnp.array(rng.uniform(0.5, 1.5, self.n_q), requires_grad=True)
        out_bi = pnp.array(0.0, requires_grad=True)
        return var_w, reup_w, out_sc, out_bi

    def _pred_single(self, x, var_w, reup_w, out_sc, out_bi):
        exps = vqr_f2(x, var_w, reup_w)
        return pnp.sum(pnp.array(exps) * out_sc) + out_bi

    def bootstrap_predict(self, X_tr, y_tr, X_te, verbose=True):
        cfg = self.cfg
        B   = cfg["n_bootstrap"]
        preds_mat = np.zeros((B, len(X_te)))
        loss_hist = []
        y_log = np.log1p(y_tr).astype(np.float64)
        mu_y, sig_y = y_log.mean(), y_log.std() + 1e-8
        y_norm = (y_log - mu_y) / sig_y

        for b in range(B):
            t0  = time.time()
            rng = np.random.RandomState(cfg["seed"] + b)
            idx = rng.choice(len(X_tr), size=len(X_tr), replace=True)
            Xb  = pnp.array(X_tr[idx], requires_grad=False)
            yb  = pnp.array(y_norm[idx], requires_grad=False)
            var_w, reup_w, out_sc, out_bi = self._init(rng)

            # Cosine annealing
            def lr_schedule(epoch):
                t = epoch / cfg["n_epochs"]
                return cfg["lr_min"] + 0.5 * (cfg["lr_init"] - cfg["lr_min"]) * (1 + np.cos(np.pi * t))

            best_loss, best_params, patience, hist = float("inf"), None, 0, []

            for epoch in range(cfg["n_epochs"]):
                lr  = lr_schedule(epoch)
                opt = qml.AdamOptimizer(stepsize=lr)

                # Mini-batch
                b_idx = rng.choice(len(Xb), size=min(cfg["batch_size"], len(Xb)), replace=False)
                Xbatch = Xb[b_idx]; ybatch = yb[b_idx]

                def cost(vw, rw, sc, bi):
                    p = pnp.array([self._pred_single(Xbatch[i], vw, rw, sc, bi)
                                   for i in range(len(Xbatch))])
                    return pnp.mean((p - ybatch) ** 2)

                (var_w, reup_w, out_sc, out_bi), loss = opt.step_and_cost(
                    cost, var_w, reup_w, out_sc, out_bi)
                lv = float(loss); hist.append(lv)
                if lv < best_loss:
                    best_loss = lv
                    best_params = (deepcopy(var_w.numpy()), deepcopy(reup_w.numpy()),
                                   deepcopy(out_sc.numpy()), float(out_bi))
                    patience = 0
                else:
                    patience += 1
                    if patience >= cfg["patience"]: break

            vw, rw, sc, bi = best_params
            vw = pnp.array(vw, requires_grad=False)
            rw = pnp.array(rw, requires_grad=False)
            sc = pnp.array(sc, requires_grad=False)
            raw = np.array([float(self._pred_single(
                pnp.array(X_te[i], requires_grad=False), vw, rw, sc, bi))
                for i in range(len(X_te))])
            preds_mat[b] = np.maximum(np.expm1(raw * sig_y + mu_y), 0)
            loss_hist.append(hist)
            if verbose:
                print(f"  réplica {b+1}/{B}: {len(hist)} épocas | loss={best_loss:.4f} | {time.time()-t0:.0f}s")

        return np.median(preds_mat, axis=0), preds_mat, loss_hist

print("[OK] VQRFase2 definida")

[OK] VQRFase2 definida


In [7]:
RESULTADOS = {}
for cen, dados in CENARIOS_F2.items():
    print(f"\n{'='*65}\n  CENÁRIO {cen}\n{'='*65}")
    t_start = time.time()
    vqr = VQRFase2(CONFIG)
    med, preds_matrix, loss_hist = vqr.bootstrap_predict(
        dados["X_train"], dados["y_train"], dados["X_test"])
    m = metricas(dados["y_test"], med, preds_matrix, nome=f"VQR_F2_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds_matrix, "mediana": med,
                        "y_test": dados["y_test"], "loss_hist": loss_hist,
                        "tempo_s": time.time() - t_start}
    print(f"  R²={m['R2']:.4f} | WIS={m['WIS']:.2f} | {RESULTADOS[cen]['tempo_s']/60:.1f} min")


  CENÁRIO C1
  réplica 1/5: 28 épocas | loss=0.0909 | 61s
  réplica 2/5: 31 épocas | loss=0.0441 | 65s
  réplica 3/5: 80 épocas | loss=0.0035 | 147s
  réplica 4/5: 19 épocas | loss=0.0561 | 44s
  réplica 5/5: 80 épocas | loss=0.0028 | 154s
  R²=-0.0106 | WIS=1967.00 | 7.9 min

  CENÁRIO C2
  réplica 1/5: 71 épocas | loss=0.0138 | 132s
  réplica 2/5: 80 épocas | loss=0.0085 | 149s
  réplica 3/5: 23 épocas | loss=0.1075 | 46s
  réplica 4/5: 80 épocas | loss=0.0085 | 146s
  réplica 5/5: 33 épocas | loss=0.1356 | 64s
  R²=-0.1528 | WIS=3168.59 | 9.0 min

  CENÁRIO C3
  réplica 1/5: 45 épocas | loss=0.0331 | 84s
  réplica 2/5: 24 épocas | loss=0.1049 | 47s
  réplica 3/5: 32 épocas | loss=0.1022 | 61s
  réplica 4/5: 26 épocas | loss=0.1259 | 46s
  réplica 5/5: 80 épocas | loss=0.0358 | 145s
  R²=-6.9234 | WIS=294.19 | 6.4 min


In [8]:
print(f"\n{'='*70}")
print(f"{'FASE 2 — VQR Data Re-uploading (4q, 6L, 3 re-up)':^70}")
print(f"{'='*70}")
print(f"{'Cenário':<10} {'R²':>8} {'RMSE':>10} {'WIS':>10} {'WIS_norm':>10}")
print("-" * 70)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10} {r['R2']:>8.4f} {r['RMSE']:>10.1f} {r['WIS']:>10.2f} {r['WIS_norm']:>10.4f}")
print("=" * 70)
print("Re-uploading permite ao circuito reprocessar os dados em múltiplas profundidades,")
print("capturando regimes epidemiológicos distintos (endêmico ↔ surto ↔ pós-surto).")

plot_pred(RESULTADOS, "Fase 2 — VQR Re-uploading: Predição vs. Observado", "fase2_pred_vs_obs.png")
OUTFILE = "fase2_resultados.json"


           FASE 2 — VQR Data Re-uploading (4q, 6L, 3 re-up)           
Cenário          R²       RMSE        WIS   WIS_norm
----------------------------------------------------------------------
C1          -0.0106     5753.1    1967.00     0.6958
C2          -0.1528     7448.1    3168.59     0.8085
C3          -6.9234      532.1     294.19     0.5531
Re-uploading permite ao circuito reprocessar os dados em múltiplas profundidades,
capturando regimes epidemiológicos distintos (endêmico ↔ surto ↔ pós-surto).
[SALVO] fase2_pred_vs_obs.png


In [9]:
import json as _json
resumo = {}
for cen, r in RESULTADOS.items():
    resumo[cen] = {k: round(float(v), 4) for k, v in r.items() if isinstance(v, float)}
with open(OUTFILE, "w") as f:
    _json.dump(resumo, f, indent=2)
print(f"[SALVO] {OUTFILE}")

[SALVO] fase2_resultados.json


In [10]:
import json as _json, os as _os

## Justificativa dos Hiperparâmetros — VQR Data Re-uploading

| Hiperparâmetro | Valor | Justificativa | Referência |
|---|---|---|---|
| `n_qubits` | 6 | 2⁶ = 64 dim. de Hilbert; padrão NISQ; AmplitudeEmbedding exige exatamente 2^n features | Preskill (2018). *Quantum Computing in the NISQ Era*. Quantum, 2, 79 |
| `n_layers` | 6 | Profundidade necessária para aproximação universal com re-uploading; cada re-uploading consome 1 camada do total | Pérez-Salinas et al. (2020). *Data re-uploading for a universal quantum classifier*. Quantum, 4, 226 |
| `reuploading_layers` | 3 | Pérez-Salinas et al. (2020) demonstram que 3 re-uploadings são suficientes para aproximar qualquer função contínua com precisão ε; aumentar para 4+ traz ganho marginal com custo quadrático | Pérez-Salinas et al. (2020). Quantum, 4, 226 |
| `n_features` | 64 | AmplitudeEmbedding codifica o vetor de estado de 2^n_qubits amplitudes; 13 features reais + 51 zeros de padding — a normalização garante que o padding não distorce a representação | Schuld & Petruccione (2021). Springer |
| `lr_init` | 0.05 | Ponto de partida do cosine annealing; valor alto permite exploração ampla nas primeiras épocas | Loshchilov & Hutter (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts*. ICLR |
| `lr_min` | 0.001 | Piso do annealing; evita oscilações tardias e permite refinamento fino dos parâmetros | Loshchilov & Hutter (2017). ICLR |
| `batch_size` | 16 | Mini-batch balanceia estabilidade do gradiente e tempo por época; para N~80 amostras, batch=16 ≈ 20% dos dados — recomendado para séries epidemiológicas curtas | Smith (2018). *A disciplined approach to neural network hyper-parameters*. arXiv:1803.09820 |
| `n_epochs` | 80 | Orçamento máximo com early stopping (patience=15); suficiente para convergência observada em experimentos preliminares | — |
| `n_bootstrap` | 5 | Protocolo WIS — ver Fase 0 | Bracher et al. (2021) |
| `patience` | 15 | Early stopping — ver Fase 1 | Prechelt (1998) |
| `seed` | 42 | Reprodutibilidade | — |

In [11]:
# var: n_layers×n_q×3 | reup: n_reuploading×n_q×3 | out: n_q+1
SCHEMA_INFO = {
    "algoritmo": "VQR-ReUploading",
    "fase": 2,
    "tipo": "variacional",
    "n_parametros_quanticos": (CONFIG["n_layers"] * CONFIG["n_qubits"] * 3
                               + CONFIG["reuploading_layers"] * CONFIG["n_qubits"] * 3
                               + CONFIG["n_qubits"] + 1),
    "config": {k: v for k, v in CONFIG.items() if k != "quantis"},
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Fase2_VQR_ReUploading")
validar_golden(doc, contexto="Fase2_VQR_ReUploading")
# ── MLflow: registro automático do experimento ────────────────────────────────
_MLFLOW = False  # tracking desativado (entregável)
if _MLFLOW:
    with mlflow.start_run(run_name="Fase2_VQR_ReUploading"):
        mlflow.log_params(SCHEMA_INFO.get("config", {}))
        mlflow.log_param("algoritmo",  SCHEMA_INFO.get("algoritmo", ""))
        mlflow.log_param("fase",       SCHEMA_INFO.get("fase", 0))
        mlflow.log_param("tipo",       SCHEMA_INFO.get("tipo", ""))
        for _cen in ["C1", "C2", "C3"]:
            if _cen in doc:
                mlflow.log_metric(f"WIS_{_cen}",      doc[_cen].get("WIS", float("nan")))
                mlflow.log_metric(f"WIS_norm_{_cen}", doc[_cen].get("WIS_norm", float("nan")))
                mlflow.log_metric(f"R2_{_cen}",       doc[_cen].get("R2", float("nan")))
                mlflow.log_metric(f"RMSE_{_cen}",     doc[_cen].get("RMSE", float("nan")))


[PADRAO] fase02_vqr-reuploading_resultados.json
  Algoritmo : VQR-ReUploading
  Tipo      : variacional
  Parametros quanticos: 169
  C1: R2=-0.0106 | WIS=1967.00 | 471.3s
  C2: R2=-0.1528 | WIS=3168.59 | 537.1s
  C3: R2=-6.9234 | WIS=294.19 | 381.9s
[CONTRATO OK] [Fase2_VQR_ReUploading] JSON valido — todos os campos e invariantes corretos
[GOLDEN OK] [Fase2_VQR_ReUploading] Resultados dentro da tolerancia vs. referencia.
